# Batch-50 optimization experiments

Choose **T4 GPU**, then **Run all**. This self-contained notebook stages the frozen source/data and tests bounded vocabulary projection/loss, midpoint force reuse, and a correctness-gated fused AdamW option. Custom kernel work is parked.

The memory safety margin is fixed at **10% of total GPU memory**, applied to **peak reserved** memory during startup, training and checkpoint staging. Physical batches 29, 40 and 50 use accumulation 2. Batch 50 therefore has effective batch 100 versus the baseline's 58; these short probes do not establish equal-quality training savings.

Numerical gates precede the sweep. References bracket candidates; the best feasible batch-50 configuration per method receives three fresh repeats. Acceptance requires every repeat to fit and the slowest finalist to match or exceed the fastest same-session baseline-29 reference. Historical throughput is also checked separately. Optimized baseline and midpoint are both reported.

CPU checkpoint snapshots retain optimizer/scaler/RNG/cursor state, measure CPU-copy and disk-write overhead separately, and are marked **probe-only**. Original full-run checkpoints are preserved. This notebook does not launch a new 50M training run.


In [ ]:
import importlib.metadata, subprocess, sys
# Match the recorded Torch/CUDA build and NumPy. Python differences are recorded below.
required = {"torch": "2.11.0+cu128", "numpy": "2.1.3"}
wrong = []
for name, version in required.items():
    try:
        if importlib.metadata.version(name) != version: wrong.append(name)
    except importlib.metadata.PackageNotFoundError:
        wrong.append(name)
if wrong:
    if "torch" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cu128",
                        "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    if "numpy" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3"], check=True)
    imported = [name for name in wrong if name in sys.modules]
    if imported:
        raise RuntimeError("Packages changed after import. Choose Runtime → Restart session, then run all cells again.")
print("Pinned packages installed. If Colab requests a restart, restart the session and rerun these cells.")


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, uuid, platform, torch, numpy as np

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
if torch.cuda.get_device_name(0) != "Tesla T4":
    raise RuntimeError("Select T4 for the current bottleneck comparison. Another GPU would be a separately declared experiment.")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
DRIVE_ROOT = Path("/content/drive/MyDrive/Reversability")
OUTPUT = DRIVE_ROOT / "optimization-v1" / RUN_ID
REPO = Path("/content/reversability-optimize-" + RUN_ID)
PINNED_COMMIT = "d071e1e2d8e4575d2fc9ca5ccb1ef4e3d5e185b7"
subprocess.run(["git", "clone", "https://github.com/LokeshJatangi/Reversability.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO, check=True)
manifest = json.loads((REPO / "results/recorded/reversible-v1/source-manifest.json").read_text())
for relative, expected in manifest.items():
    if hashlib.sha256((REPO / relative).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Frozen source differs: " + relative)
reference_config = REPO / "results/recorded/configs/baseline_colab.json"
reference_metrics = REPO / "results/recorded/runs/baseline_20m_fineweb_edu_50m_v1/metrics.jsonl"
startup = next(json.loads(line) for line in reference_metrics.read_text().splitlines()
               if json.loads(line).get("event") == "startup")
if hashlib.sha256(reference_config.read_bytes()).hexdigest() != startup["config_sha256"]:
    raise RuntimeError("Recorded baseline configuration checksum differs")
actual = {"torch": torch.__version__, "numpy": np.__version__, "cuda_runtime": torch.version.cuda,
          "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0)}
expected = {"torch": startup["environment"]["torch"], "numpy": startup["environment"]["numpy"],
            "cuda_runtime": startup["environment"]["cuda_runtime"],
            "python": startup["environment"]["python"].split()[0],
            "gpu": startup["environment"]["accelerator"]["name"]}
mismatches = {key: {"recorded": expected[key], "actual": actual[key]}
              for key in actual if actual[key] != expected[key]}
if any(key != "python" for key in mismatches):
    raise RuntimeError("Runtime differs from the intended comparison: " + json.dumps(mismatches))
# Python patch/version differences are declared; these are new paired probes, not a rerun claim.
print("Runtime:", actual)
print("Declared differences:", mismatches)
print("New evidence destination:", OUTPUT)


## Stage the original dataset

The default points to the original Drive artifact root. The existing staging tool also checks restored `baseline-recovery-*` folders and recovery ZIPs. If it reports multiple candidates, set `ARTIFACTS` to the intended artifact root and rerun this cell. It verifies the original manifest, train/validation binaries and tokenizer hashes, and copies data to local runtime storage. It never downloads or prepares replacement training data.


In [ ]:
ARTIFACTS = DRIVE_ROOT / "artifacts"  # Change only if staging reports ambiguous candidates.
LOCAL_DATA = Path("/content/reversibility-optimization-data") / "data_fineweb_edu_gpt2_50m_v1"
sys.path.insert(0, str(REPO / "scripts"))
from stage_baseline_data import stage_data
receipt = stage_data(ARTIFACTS, LOCAL_DATA, DRIVE_ROOT)
if receipt["manifest_sha256"] != "71a12f5765cf37b7f5fb8753d51fa0828915a6db313ae20aef7fa9fb2e14df8a":
    raise RuntimeError("Dataset is not the frozen study stream")
print(json.dumps(receipt, indent=2))


In [ ]:
import base64, zlib
EMBEDDED = {'scripts/profile_training.py': {'sha256': '2251d956655aafbf86845e89ca543ef7d000ca09fb8236125c6da081c0fb4aba', 'compressed': 'eJzdPGlz3DaW3/UrEOaD2UqLumI7lqenxknsVKrGtsrHTtVqVSw2iVYz4jUEKanj0X/fdwAgeHRLTrJftqtsdRPAw8PDu/HAb785bFV9uEyLQ1nciGrTrMvidM/zvI+yiuqokaJJ87S4mouykvC7rEVTR7FUIioSEWVZGdPDJI2uilI1aaxeikLeyFrUUrW5xO5pARACALq3qstchOGqbdpahqFI86qsG4BVlE3UpGWh9vbMs/oKMFDS/I7LopF3TZYuGUoCyAFu0sAwv+eIsfy9LOzIVVvETVlmyjxYR2qNcPTP31RZmO951KwZfgXfoJMBf44N+nsNay9z8wtgyTv7A5dBZLBP2mVVl0Cx7snGfkVU7YqLNq82IlKiqGx7Wcfr3o+gKAJaEBAryrD3m729D+/ffxILwtEH6qYZ0HYWwAaU2Y30ZwEQUhaNuji+3IPJA1xakBZK1o1/NAeca58gHApP1bE3mzEFatxGlS7TLG02hg7LNs2SMC8TmVlyuP0C8yuTiFz3i2ESN4TLSMksLezeLaMmXodRMxeEu1pHJ0+fzUXdFkigMJdNBNsb7e29+fD+v1+/C9++evfrm9cfcc3e8+Po+GT19Pmzp/Hq9Pny+erpavnD86enydPjVXT0w8kPL46fRs+S5enxaSRPjiK5er6KXqyWJ/L4+2T1Q+Tt7e0lciVUBKRCyszFTZS1cna2J+BDNMXHs+C2ThsZIhP6yDJBAvulfOo8F2mRAIkXJ3MSituwiIrFmyhTcia+E97/FN7MzLMpYj+RN2lspkhXgn8HzaaSYgGLitsk8rgVP7zz+DDA4eu6LNLfpYECgOMsUkp8kFcoQjwOxO2dVI1MRCWjawV8WN+kN1LEa9hAFE7ZgKg64BJXiEH8pAChFEsJHfKovlYkwAgZVxGGINRNGPqwkau5xn8ukC9mHd7YGJg2+oEdYNfc/v3edXmrdN8ImPwGe19czuFfvx8uKtT6B1BfOE9xbfUNPTzasxirKK8ySfg6GALtHSR5A74ZbwB+aglKq7CP3Lmd/cmjO+DXvKw3HXa+M8Vsr4Nn0ZwebzpMD5+mAgz3J1rmHbpTEBxU+gBMw9xi2w1fgdqH3QLOd7drQLPy9sLroxIuN41U3qWea0ePSaT7UA1WW4EOO7gLIZj/6OxKoL/mURFdydpyTk1ipTm9iHI5wT/M18A4bC4HrLNJZZbsYqYxGDSyAyi3abPWrAIWBTVlDdo2LuskNBbB93TLoQdqZ4Dro7EhFbWd6bQkOZwMXLAQXzwmlHdGE8/F9LaeiSPTNNgcaLnfKZgTmtGyAxjeupnkMle4HiWYA5gTPDYGuVtW8eOMIeUbEgn0cHQb1A6Ss2wFUVXJIkH27tqberOL2VYpuAnZoMv2Dd6+yUNcqrIatH7FllkSw3L+uk3rQf2Ltu3PbJ0lGRq13ubt7e3QO6hzHI/YJ2drrtWQmlnz/kmi9xTVG3FbI/BaibKArRaVXpZo1lKUdXqFPCDA//pdFmRys8NlFF/fRnUCbnUirWVvwa8le2t1H0L2y9tC1nMt1Vm0lJmjVSz8hbgCP60BZ9Lp3xEDHobYmi7bBo06tqLtoM5BGCZpDP6E7f4P67UHiIPyzTwdRIdOsIP7EDCA57C/j+u6UgO9B0zbRllIyMPk9NefIcvGwFgR+Kc+L0xI8Nm4fax89RYE2iK4QCf0LGtVS6ARho6Ij+g271bWXzBsLbAjCObZUPZ6BB5jMznJmKj4QRKMAQDbbN9d5BzL4YwiN/bUE7oMSxDja9x4YsSAfqqBoUOmo4YAZizAXMBA5FYPvsYlcH0BvvYhtMEf2IjQNM+2gcmzahsUaJoYT2MZw6a8lkUo86VMEoqEp+EMuk3DqkqVEsYPghv3nIZI+j0syjrfBqnrMQ0hy8O1jJJtw8u2qVpQeXX5myQfw4ECXGdkXispzygWChEB0juAMiMzgK5KAbFmGzdDxwaR6SJFgJIW+EOGVZTWiE7XeGiaHHB9pKYh6rUZiFmUL5PorAcZ7UtGcHNYsfRGEoDrZVuQqvCqjpJQFqg6El+rDRca+GVyF2MyoKhtSoQEGOKfwVoZo5vfKgdAZ95Hpp08chY+FDAGBNihcM6GwYxWIzoqLSt0XH+XddhWmEkx+2mfQ0wGuOBfCuHnmI9ZpcC+cVursu5CwL6ZosgeFC73Bqd8vVEpLopDfgWgPbY2mSyuGrerkv9uIfiUIbfoblEct3mbkWl0OiPxUmDX0G0H2ywrpQfGZVs0MIIx2jfz7fcg8hR5Bf18wyLdRsWRanx2Z2C96N8svnir6viZZ3qtsjJqjp/Ne7T2liuny1L3ub+wNAGWSxUy8eXlbLY1HUAZt4kxFK+uqtMTj5nQcSyKNsv0T8Joyoh5vOGHsMclMbXnGnez/YFt9tEBakpQKIVcfKpbreezUil43JAjcBQc0UOd+NKhuGHRkJgzKq6k79LemXYXnpgICoEUIfBhoVay9gasfTcXG8rhpInZbrCLvmFbYlcIjixuc+4z1xzhdCGWmY2AA9Q7sAom/QJPnF/9NQArDS31rrWR0HVKY2yDs/IqbRRGudjTv5s9GnRc4waBhNRltZkGrRDwm6DX0+cZgbHJ2mK68JhWbB7M5mM1Of1JrwpUiZgou1scHB8doa5IWH0vPNXm3tjzRqsFG/k7cRBheMi78uh1G3s0tWQMWEipiVSRoRp36SMRGGj+GNVppwk/PEdAf/wO2GwK2rfiLekn9NyJY2X9RBGEqBYgxnVE9DrM0hVnv5dyheoe+xfQLvIU9w/0XvAoaSKxZfBDCjkS/R0THwKcJorX/oz1HPxNG5k72HfyDiNIAh9UO21BdCGJjrO08gbJFbs/mIwc75GmrYHiW4XVIYUUt8FfUQRtk0JQgXOxDcf20PhnUQ1ubQNmExi7Z1wYucvZgwtS8Rq4Wq/IouMuC6h0BfYXpc0f6Bpib54UE4GcLW8gWpCNsWM0f1TnLRqpPC3842C+Cw737YB0hInBxSykY0QBGhjbqMbjmhAPf8C4fCeCp2AkfdNn2H6wa/QMRx4DDDxWgTAXYln8UqXw3FBh1iGU1XpJ2ybb1yufa9z7ycirugSiYEhprRbtZ0gNgxiDnsEMNWUGsnqa6cYM18FGIXN4fyz/WjQXhkmB/iErgdkUDxO8Cf512Zx9s3HaZTSB+JuefiIsjVLwEt6gBANpz8u0aF7XdVn73qu356IEnxGk+xazCJhJhC6CpwXagya6TiHIS16CVIHOARpercFXFnEWpUZ/67D3i4cqA9weR49o5a2dXfLXKC45IyG16gXaDbuedTrlXruqIF/xtXFT8ZAxa7qDFNQSxGOpWuEJhelxwdgARwKj0LlGWWa+ceYHXfvIgU/WEXE78d4hzyIYWvChgYH82FESvi2lPQrqZZB2HQlNnEhYKm9JsiLnbs2zYuP93mPAPHSyYXygvsM7PesDpxwalNlmCF5gB+ry1qdvmjAWX0w/AExqC67lhmJW4C37jDmtj1dctcyJYau6jvgQjxmpBeBgyi7Ep04n+8zp2YfN6LvgTWhMEOZdDwsBooijYEg7mkl3nQTjdvgKWJrcYOXAfzZbsgu2HtCq6AoD+aMh8LTAvIBaR5Xs6OQ+NDt5W9bXsvadrBzrd9CMdI4KwpQoag64AUwqGFw6aNXWQT83sals1mWyoBH8fS4mAkvuwM79hGNGgUSS1gs8/6au5gmZ/rxKM6mPcI1MOjNSXJanSYUqwJHLPqa9bMiilwLBc3dZMaLB0/4UyGgqL6/lEG4/8DNxn87W4q5ZR4d/+sjy3hiuTleQTplNaZx+1IkK05HeVIXRTZRy2nSkHD/wqb1Wiz99/vmV+OX8M0juv9sUHP5AHBwAHgeMB9jYKmvzJSpHchg5sMc8dqAxI8w5CxXk17A9vi5noOhzDl5xqpqwvHb3ijU5HUqhdiU58uq2QGcCcyyUnDrT+wnfOeOVw1rhKRV0BL/BxvpYLQGdbkZRDp7zSQCdV2HbxDDIlJ8EBWgsU4ECvmYMTrIqV+j1N+NgyeNthfEmnaKfhE4hBLQ6v1xBGcOTxU1al0UuSRMOCyi6kNWzrBAirQ0t7FO2DlQT4dAfK0SYtAFKrjfXhJ5I9WJmfYJlkZPAIvqcNpnr3MhcM/IwSUX89F9YYKG56XOh2gpLRrCswYKdTc06Sh+Jv3euKjr73iU72aOODyHxE2dUgPFiKRMlyEtFhHTKVtiUrU4p9BFEEgykcVtmB6WQCLRFCrEttBQZZhrGmP/45viZaB0aArLNGmQQBNRBEuQAwksQKqx7ASHqKUfkAdOBuWA0rq/Ye9DGqr0T1wsL1zA+qjiX9XugHkPU7YDBpRoUEz1EvHdIfgiz9TmaatpkQ0ZEycYu3aHiNuJRUBeAxvPcBRBxUFdBVI14gllAHO2SLzxTSNeFchBYHWOIdYJOrUsnIg8O70Zjs0KWJzD4xZDioXV/0vMKQBkPGxG1Qyyhg1BN5WhdnUUTdDzdq/CINY8qXVLF2VJ42oK5PH7GlT9gEJ2hlKw1BGP7qTqjO6ivYANGTVQqEW4fQJUU3P+4w5NpGFY2421mNvnh7Xpk/yuyznp3J6aDyJjm07H8d7z+GWgpjIB3bft8V47gQR1wjiEI+GRtlmgNRlxtDbrhbzOvwd3ZKa5/BJ8YdE5HJpnA7C9x53e2swaD9eHR6VQXl26PKCXoRRUWKIYoWwGT+sfccFfLqPvOnExuQBTwx1qqSxJxwVE/cxRS4DExTJVtHUtXtX1BzxPIhQer6Q068lSHCfFmX5pnlN7A1EZv5Re2aDOu06pRh7r2J7Qlt9UGTOuwV78Ek/vsK7Ygbh3oYa+u05sFV1m59L19HDGbXXalOjZnYZ1PehK8SqL8X5OJtaxebEnxTDnqS/Be1MLuJv48dqUAH5zA/s7FrUyv1g0EMHG0sTO4D10u0FkegzO4csEvIM8f6bE+5vkDpzEL8srRZpPKwVC7MwkZa0ddqOmbMzOj1ZzcHOfzzLHJHxKHbbUqw5OK4ZlMp5UGyqSXefnrTgyJKrN+UsvkMxcTqrPbQaeUdiRuvAKWM52bVcbNNVlEJ81kkNrfn6LV/f/9LjDeqiu/0bNNmr/+bElbc6lQf/DE3rKNmSgOcSj5UlDdGwoGxjNA9xXrNJCK8bEHngBA17+aGx7A0K7Y1JxMoCoOeCFjWD0+xgUARE1+Aw+f2pzhGMBjuNNlRr1tLjOCmQISJMh7djXIkZQ/6tpUm/u2feKwzctlVLj97QWEAFvcsdg3Sbf3xrbdc6kGdmB6ODU5o5F1wWNyn/xdHLNOPAomYLuU5BloAvQJ96f8p8NHkUa1NTgDlFkIO2HfDvZhBQBWc9rFHBWcpTcQDkoSykHV7rn+8oo7bYKfzj/35fZxaqY/kZWFhyb7/POrQbUr26NJy0Rrmzhn3lGhOB9WKesvfofrovs6fYCsC5s5k6gzPca/4Q3SDxEZ3Kv4mqsR8NaJ7jgm1iPVIe3AX63YDHUmZ/vTKomAPFYt4WfSeOJHEy+Qd5gkCPF6SE4+ZYwEq4f5IGrgRMDAhlNGmNjfyeWzLyu7YxGJ2fswAlczupLKp3O5cLkJnUwyb2xfQKZSU+aKmjLZKcZgyOuskvV2kE6+kB1WGmsQQHnBpwyXAdkRhZzmFNgPEUJiwVIwK7slRb/dmeR07MQhQ78IbfuCm7vG691Q+kriBg0lc2kRy83CLGaO9dRhBpzeLJ4ezaYpyQH4OlIhLiZkAhJdo2Ljy8AQQi+5X/gW/Eytn6CRFNOYPRieP5j8wXz1A9htOxedyF5fy7qADYLoQt+JiqMK7y8mLw2O9eFPn88//YrZ7LawkbQ3Ta6UTgAwQwuWiVUZx5G7vMPx6fIOna2BDqdHhYoB/UJQprzvsA5vEeyi8bTCHFXXB6FW4npl6xRrDjc+Zi2wvAjNwPERfqbVmYsxMumo00N26P+FYu87vqboZPJ6ht4zS7dH7xPeZAxVEVVqXTZTSt5e9LW9giqNr5HH+yhMXjf5w3hN848uul1QafGkiNmj5U7F22qd8lbtGtPdgaSYTbvsW07HDdDBNbzdap5O5icpsOuz5Uy9h4G9xvcwAp0/q89zNIfyadmC6r8z2eDBLzImSKpm2rmwXrUOqhfkRfVX1HO4HS9kMeVxy9VK0p0ic3arM65qMZmQ3Zl07aMBmkEuvDeA+ppLL+ZYtyLxyi6NOiTppCNc8evh+5fGb6Lbr8rcgKeLy4YgiVi1WYYnayoArQS2BU0kj0iLOGsV3llFI4SVNFlUBeItcZbeKn0bF7tzTUxG/Zl3oH3OFsaU3x/gbYuEnXw+GqWdD7xxwoMrNcH6F6CdNN+ShQc2uXCOwrdntufjA7IbQGupy7QHxKUsl2XEr86dT06BhxnDGgZwwdhTCfGM8K+f8HuWBcyFV414TX+Q0BBHSPQCzh4QlRWYe4mV+9Qb5KSqffrqqMavO0iludAb+QOHsFUNzOzeT3dOwG0mmh/AfnsMkkLintKfTRyS38PDFTD4Wpd96/vsLZZOOcUdf+S8XsWg1Ou0pGjB90xyOjx5QWfE+id8P3kBSPi28EJ3sHUY4w6np4MOp6facd9RbLEVnSE2A2SGuAxQGWKiEWEr5yTv0AXm+1mmuoX5HG9aG8z6tz4oFUHn6jrTZCN3xx0cJA0lGLnC3KPo8xfdzTvs387Hj66RQEyxMkLeybilwAFmOWg9fpXEtrdQIEsdHOiCh3nHcFtKGQT2xnPTg4Ru7ExV6UAPzcDc7qyJW5l+niXk1BxEWw2AvuuhfH9JV4ccHOh8soOIzpFPAjUJZ6e7fsTQaXOmevVOM2ejpNA2jh3skHEtPafWZhAPOLTS7wfZLa3dVQ/zYVWz8s5teeMXZJz7M/GF1IXwV8byYgXnzOurDxcUefL+GCVgwpXH0AKwbxDiQshb+N6tR3keeDRFAnrFRPfelQCMta8JAybuNlngKRfSPAHuWQAQ+g46e+EM+vjp5/efPyH/YsVhPBQF7ZrqQ35XmAbqeWJUvz7CgdSrjuhcZm6jnVDmqplb2GQtkEd2Bx+9K3XZnVb6L0E7qArcLKSZdz+BlA1HALJWMt1Nfm0KtDB0JKHCJf0MpyEjg9DuB/XLEzaMDIcxYVvqtB6oUprzrcI6UXZidT/NpEJ84Q4XT3imJ5fIqM7G3U/wp/H6UENmmV9fuGuns74jrjO5MAS47Mwr+93GkebbeXwvjxCdbbXuWynTAaNCJ40bJ49AYJo0ykJkBnDP/xz10L6RQfK+Fab+44B43dYy4wT476fzz4ILbnASKgynrAmodfKGMTrZXoFIXck1Vy9tycEBQtIvnlLomGUtiHTntvPZa891Dww6/xEfNfeK/2DdQ5oYp/5QwZO3dMphKtr1SQY8/9iA5G2cl7r8kv7YPbaBFT2laQ4ODvDf2fA/rzPh7mb3CuNqdkb11T1T0YIjJtnoG5eNBieyZQHs66RG8NiuB8WcPNkeV+mSLsDj5UZxd4b69Q6Y7mR///ToLDhd3dNm3blXbvReuaktC444xaiOFZD/Sw0SprcAZAyI+KW5eDI+3XlyeRYcr0x7/2QK277nNkDXh/apAPzJ5WzUpR8gcw/P5envkKlxB+meFWwpmfyDkxeaC2HdZUbvPVtL7TWQpMFyUGWLH8kTOz0VLQTFXeiqPbRnz8Bxh5E1DAc2e/oDsSqQMurCSffaBLk++gjLsvC33+rAEsey8wac1Mi6qiVnukzP9+adbJg5PtAVwHQtrI4KiDJt5hLMK6gXEXFVYsHvherCVZNFEDsu1+uCh4lr8wwooRw6/saXYdHNY12Cx/epWXLxbQsKqahfP2EwrGWOpRE3Kb85LC343RW9d869NBPw6Y0SSakzvfTCC1AEAFxYxA/f/vNc2GsYdEBs6PbgdXLMevQug9NS7HXsXn4AVBW+HwsIimQGzE3RdyB+tRSmcmGTEmB3x9LSLIe2u1WNfu8Wnm/mWC/N+V3DCZw/MCTHFXPEr3QmAYsUI17vS0MhgIT0zFHHxDQEf3RJBVa4YIpQ4VoqfYYdbAs3Xa11sslhKNqkWq5kjUF3AOoSln5Iwbpz6QW5FqDw9UWNbTdKkWviTEGszzPEIENL5Aa8bYrlqrVA84aLjLD+9yYtWwWYr9KmoQpFFMJAvLKvJTTZSiZu72Vn1RriuANDVJZ93NYuAcipGL1k9z1omj6/ao9q9IYEcDLdO7RzfW33gAgDQhSBH3cIzw5UuWry6E4sW9AjtZpb1hG/vH77Vln56RjTXCqL1yXegLuCLiZjzUpEvALdDroW9zlumWz61rCmfJShSk9KyQoe0Aa2SRt4kqMnJJVWfZzlo50wS155H9E1EWyawH16YuzSk7FH8mTgkXQuKPurHLQCbmrevbFGtTF636s202RVT+4DMEKcdbM1ieAiitK8YrLnC+AlNQCcrlIQHW32hv7Vh9fn7z98CvKkf2SGb+njsn8yFd2L+7TZNok5XOLuSw8fy9wVgCrCDKpe9ku7WFOerLRyuVJ0eZHRhEgMSMXHOJxoobdhhrgUU+hdceBODcGr+opE6Bx/UShM9X54uToMkzKGWJwHBWAFEAz19t2QnGp0z6le19zWcHzh8UAnOv/aoTZs/9qB5BvDuMjcG8fXWIQgHia+HQ+xSQAQmhTTyhduDqfLyFzikc4qgh1ZdB22ATVZA8I/xctTZuzJi62IcEqhQ2N7ssbFpVccOIZqExNjVJ5uG9MlHibwP9o2api0GI893spjNgsxuXVzsZZZtfA+pcVGYEQxcSnIvFcWLxOB5k/QeJgrQiAIVeAKCD8O7EWzqLtaptNRuoetzF84P0a9tFAu7NdBD9AOWLUd2crCyKR78Gsvp4MPOM0k/iaOnVvhgWTl8S8NQscoVM/GKoJ9THIRwCrydZMbQ4PJO3VbL9RphB9xFLDzuGXbGswVGXpJsPvGli3I23s7eIfZ9/Ttu96l+Q40tSWOftWhimYVmXRAo8F9EH90GKDrv+l60LPvyUHacYHo2feTKNkQWKsw9PcbZGVzZFyYuyNoxSjm5Vl7N6YjUPQpvloVky5hSOmDMES/OAy9LreOvD7kdCcJb0Ntesam2L2Bufe/QPXSNg=='}, 'scripts/benchmark_optimized.py': {'sha256': 'bbfb6adbaa669bd0aff8e54c3385cf461b2e1673c0c39257820a4e7482711ed4', 'compressed': 'eJytPP1v20ayv/uv2LDoC+lQtKzEuUYpC+Ra5y5AmwSOew84P4FHiSuLZ4pk+WHHdf2/33zsLpcU5Ti9JyAxRe7Ozs73zA71zZOjtq6Olml+JPNrUd42myJ/fuA4zo9FVclVk8u6nlzGjUxEvSmqRhRlk27T3+MmLXIhP5eySrcyb+rXIi8aEYt1m2WTporTPM0vRdXmAQA7SLclTo6ryzKuaqm/r4ry9mBdFVuRwBIAWAr1RH/3Bf7/e5GbOf+ui1xfb+Nmw/NLuMrSpZ7+ER+o6yrOk2Krv9UNoF436ao2d9plWRUr2Ki5c2sucXVzXcUruYxXV/pG3m7LWxHXIi/NmKJabXpfgjwP1m2+QoLFGY5+e3D24cO5CAlLN4rWaSajyAsqWRfZtXS9AIiENL04XhwALgFuLkjzWlaNO/VhC5VLEI6EU1crx/OYBpW8llWdLtMsbW41JZZtmiXRtkhkNjIqUOwE9moWbctoVeSN/Nz4YrVp8yuZRLjpm7hK4E7ZRnUelyAMjaJ8VdAGDM8VoLdnH/55+j765c37d29PP50D2vE1sLO+zVe+2MptUd1GyIzaF2fyEogDFxqbKmpLlABegSADDrXM0txIyDJuVpsoBixp+XoTz05e+ihxyLNoK5sYQMS+iJtim64iXJ7hXcdZitAjTYrMAP3w8fzdL+/+eXoWnX/4+fTszfsfTz/54tePP705P43OTn9+c/7uH6fRz7ODg7+fvvkJmPALsDE4ngrxjXibfgYyAhNldS2TCW8Rdr2WwI0tiH6a+yLHJUXT5jC0KeD2lQSlob2ADNd1cPD3d5/OP5y9+/HNz9H5x08A/uTl8au/BK+OX05fvTo5eTGDxQ8SuRa1lIkLrFqnl75I5HW6kt78QMCHJT6wBlw4+MVZeKCmZfDgcxbabZy3cRaNDaEl0rVaMmhuSynCUDirNokdRgA/DAdv2sCiOMt2APJ+Ouavi8oliQV5U/tbt7VMwrdxVptNyqatcrUMzQ3eJPH2f3kmqlAMMgD8dT1fZFWoF81kXKGcRhWIgLPwDcJLkJg6NMjh12N4LuwbM0DXFzcyvdw0USJX8a2Ba9/EaYwx/T/cYaLEW2/S7BzUYxVn+Jdkvtv+qq3qotJsVooJwi5bQw8DJIB/RXRZxYlbyyZqiigH8xmeV61k3q0K0BEQLI15ubmtU1g3YpWqAYizEIeiY9NvrcxXMspkftlses9wmRRsVRSvVu22zcgvgF7LsnYWBywHDZi9UDEKcatdZAlvJVSCS0OVGQbVCMWU7oAkiEikOYr0pXQftarXyeBnX9z6rO8AUlsMVxOXiCqedev6D9PE308SCxqR1+vhAIt/DprCVZuFO9Y3S2OQUs/CHbOr5YS3Q/CNCGiBMYiD105rIAfixPJhFujoC4sQVbQud0+ehLxAR0QgVi3FP+KsladVBarpfKyKpRTbtkYPnjdATNFsJHB0lYHjStAvLsHKFWtF+gaMn2xqx5gORlqkNQUN70E8u+X4WdDmdBG5Rq55dl5UWyNN4FrbJs3qYJWlJYl8hM+jURtgC0+EE7Qtg81fgtmuAaw74CK4WD1tG39mHxfp7bB8A4O2bQlzt2nuHgf+QzB4bAdA62ONfq1TSIAUDcwUQAtOQO/cPWZMTB6a7eHMY4CBAVMA67l0UaZwX+/e84zGXVYF7Ai42tkUImVED+qOV/QdcKkAASbAPvQOFZ18tdsRUeiLQbc0qrXL4yWY/4GkwLpKYoCkEV263lCYCEQnSK+NkLEZ7sYbhGxo4nu+2a3cqcXbrADjk19+LNK8Uerx5pePogA3v86KGwyLRV2CoxOgHOlW6cA34hNMAuyzAuQOYi0ILSmihngaKLQtMxBbwegRCNL2SQbRQ6YsJ0VSge0K7xyE5szZkARpI7cuSx8ZjM5qoo7AMPwTJODVVhvUD0eL5bwzB/fKc626bCDCbGAQd4z5Zwj7z28KYSw0UID3U4v4EtQIjAdEY7/LHEUOrdzRj6c+CN0qaxMMI5sUZrBTrY+2cX0FsREB/gQ6TWmGuC5W8RJgV7dHiSybzWvx468/vRF//e5IRbAnxzNxyUvC8BWEPRT4MTVrJt6PH38VgHwt/joTMEAsgVoJsotBvHxBKpHBUhkwSq5a5BNP5Yjv7cfjl0LTVvw8A/KDAMXgHcTsW5853N0KptOZrygh4mWNN+CWAvjx+UxkIKVNPe+BpEEnChh9mc5eayiroqiAZBgvkzkrsnR1S3u52cC6EzWskugnryXCAzMNpN5A8oZ8oahYMUPNvlYYYfLGNgr5TTaqvAWhkSVeKDHQZlR5Hju06LwRjYHsAp52I8HbOOvy+cwhdEcjSprHNIGpd50U6/1E2QxENpjOUHtxAbQS4kROTvyewsJmkHxmHk0azJrJycudWcaNROgDgGdF1jY4HUaf9KaPpQ8XvMHFhQOJSGYHnAp8F/eaHTHHeGO7yQdCPH7p9FZWi9wrewD6miC5LhbGqMMONkWCVt11dC7lgNZv06RE4+VYcRM5AcSAFIXmuM//4nujYX/ZOoyDezydvfDFbPriO1+8mL566Xl9k8lSoa0uIxTyH1/oYIdy1dDBPYCNqNpV42DKK0sKwsLgxCcDCTmfMje9JcaSot6AokohEaOI1EqNtSCPhWWEuTEdQw3Q8PrjM7nGCLuf0uihOhTpT6nQ0u3MMQsPUqH+XJ18skfU+W1N39GW3qRA47YBwfitTSuyrmCaU7BzpYT/8iYDe5HEJXoy5QCVdTR0Vb72ePZd0BO84/44cHJgoQibT7s+Ez+UupJjBRlcs0nz5tqHkYE75AG9qSiSypCZdGDm7YJHyj+UAb1mQn8xSeqxRq5lxeF+QWHiI3KZUen5k/NJfdNVVTy8dfyQ1nI5IRSzh/X1u/0QOK1RMd3LFw8kPl+wCXvnja6dJh11sEgBZslElKSnaEbJ65uMzO127Nu4f5mqJC3xEtCENWHloCqyzJ0c++LYg4QCxMF9aNLF3BeT53MMfCfH0+noSFCtYoueGRiI7DgUuCNXrfqEZ3pB3UKgNr4Waq5dk3N1wGV86B5BIERZ2t4GIDrg+EDPK9tkubBpL1iDz2lk7k79YyyUEGrmJtxJLyFOlBEai88h4otpZdJSuBo6gLozjjl+XFudIRK16OEFJrsdnz9QOshY8a+JVwFFiLz3zB0o3FhKbRlWIIPeuG+j6Fs+cOi1u09n65Bahi0q+VbujbSi87Q7wCgnlZg+WF7b/qAJiMFJogX4PS0ND4c5rt5V7/4eEYkDXDZI0uvIJQ6BcVzu3BtX1DVl7qypab1OcwgkXQboBVjj88DTjo1Z2mP2i+7+7Oo92gMEZScldERhIuY9EmkROYhL9HwuC5FCHHJojR2mRpQ7KRrxd7AKqIqU6MvJ8cwbUdovFCb28W1fbeIrQI/z/pGwlxIkDB38RdmpmDKCJHxcDxjDfldYyQdz2q7drZXE258kXWstr3/DVB4sIXBDYzBZmksvSIp2iUoW1L+1kL+4ym7qPJc15AsKskc/dhFTEfgoUkWWPBIbGPkwPkz2kfWr4oayHTYfkAboMNlhzOAOX8AdMlAQwlzJHJP3Lxktp/OYerRyn/xEeRr9jN3pPlDjyRjkSa6lbd6e2cN0TGlj3/JP+sYcLMcSKHfUH8Q3d/Vzz8J7MzpEXNuEydLrVIHh42PvvxS1PSh9IQukel39WwUBgK0yR4iRkVUfdv186nn3Y+IEyXiMJR0q1aH1xXtXC/F9KK5pR1f+Ne6IU22S43osKlHJpbagAKU/BqI0YWU6nZtFq+CzOfCVvRlUr2hl2CxfgDyqtZy5XtUXehdz3oO1K9xDRREyjx1Q2qG8KTJ0RnlrqebvlFjPZlNKPg8e0V/fZgsXRuCRw5WSyfVsUmZtPYEkNV0BdydYCJp0xRgQ3NIxBbSNXF2R948gOH7g4Gd44rON83Qt6ybaxDWoaJKiyyuq28GRHxYDe5lr3cQVppN4GBqUslpHVAaUlbLCZXwLco7HIvaRrnun4myQOCqh4wmtjBLYIJUIberZFVprkCG6w1uBkXpP6o46q4UHUcTnrlHkOri/LF06YEjpsYsH/UHSbstaJ/Q1Dr2StzXnaQEoASCJvmEjPyfpJZCph4ChnVlwQE1AaAui1oBcdMVPRfkODD+Klm0Cf8x2Rk8EjDAVeYaygnhakNALmhXE0VFXzh89bzrcmz0d2k59/0mYtTS3dkRVjgxRB8CIc0MFcMCbmhnU8+6MmMrgZgxFIGoMRyP4HL6bMRb32yTuDaWj4N74iAPFfWfJnEHisYAFttNSlEF8uHuKwBNN7b8TzntvXFmwkhPVWHHiBHRXZyA2JI16pG5R7mx0FRs1SCwmnRmYgNoFZcPlTatFwVWK6RMUhpek9dVXoFeBgBcVnaIy5RGei+DQnJQR1rI5e8M8XR9n1ySzdk0N7RYkgSCqtXuNx39WvA40hzQnh0VBLPmpr1Y7pyneXNymEuIfembmyWx0JrJnkA2gOUcnhBadRgX0P/gkDZmbQxSCONT70jJu05Zg1rO0boZJ0c5yX1glxmxsCSRGYK5+rpjnYSZt39csUf70WlbpOiUOgUa7ceyFIV4slx4VwlEvMOA0ShGGy/5XLrHgWCY6GAhI6CEo2YlOAFEAa87bMCXTy89VmnXGPTMqwcIDkU5OIXRo8wRcXCnWcZqBp/X6XhvlCh1mU5GIoZkw5tbqzDEPl7eNpDMmimgazOEDNNFg8Xq+AzSyKSJyTiz5ZHk7PQVQpBc3eIZhjbGVxQJothHp3cNYfQmw+OAJHAGac9XKFKGDIwtGWuGLw0O7cUmbEO3jb4rqCpQRfEGthAsvg6JtyrYJtldgD1zV2UUOzBfyMwhJVFz1tY5O97Tu8leX+Y2FVwRJVNlCwK+K/2QuNV/qNmsod0AkWyCJU7U5bger/Wam8lB9cMojglg5c2w8g6fXO+cVKCoAGgLttlk5c92pBznyjaub9SBFXUEuVBcgjNuYY4eNjJOqKLbOXPdOcZxKVs+iFFpL3kWAIYDjqz3x/prqtmeFBlpBfgPVois9kN9J6yi+BvmNKWnbI/h4eMglciPm7B3Q14YUj6B214ytOs2oYFsRlea84ZTBSQfNUnnciL/n55yK9U44/IeORuyTkd45dp+z87EjuZBPjbpOD6a6wlpt2wSkVpbJqKoOFGDxCnvpsNpF9+lSnVN08TY/45B7T9lH5tdpVeTYTxoO+/i63hkVRFq2JbTtjMUbiyG8NYikihaxVRYKU6E7slyBybqagporwT/0rFdXBekhf9Hv4fQFzj1y6hVYmqY+GjZHBuWt4/eH9LsbacAhBroycV01slod9bo2IUa+zIql6xzicM9b3O8wHlP3Qb+CosAlNWSE+070VX25Y1Y/u1MepA+sS8LGFet9uwVDiwfoVKZjP/Ja1DdSlmIJ4chVT9/6XR74MWE7Og0WJJQJiOqqIxPls7UYnWfrbg9WT3tf99ODnlT1Zo2SZGCY0Qb1wD0JBx2x85HOqrd8CK8nim1ab6lQ019yjAwkR8EyHZAAm7t2Pe2T0OzowuncnU5jnh17hzMs4O448CdhNw8f1sh9AoAXWq3Gtnau+4MBxRhi4j1b05I1zN0WYY+avTm0fghJC3hnCHA51E3IJcDNFqKY45e+sp2D5SiCtizvf5tyDWlP8A/ZKHEH1DO6Vq0w3g9YrHqIFf4D2aY3Rud3ed2u1+mKWkdUU0e/AU9/7LPysUMyqgCED5+SB4SSO87DrvCF7Ou35HHuMpjYOYuxZmCF6j7TxPleyE4fwpPgb8AoOpgBqnCE9FDAMOYfQ9VpYRLRwTapbT1U7ev6aM5RUeOwLLbG3nTVHDhy8mFcNnrV8YMRzD7CPU3EXSlJVZK4yVXRTNU0bOPO3rtz2LuVvl07PIpI9ecRURTcXVrZVKp6mqMjXPWCe9wW+46XeEy/063Tk4fPkxDy0QNnSCrlwSWGaj4iVL2yB74VAL5DxlfRWAbxxU4Hy37gZkCOeMThoNkYPzvVjQ5/FQNhpaAtFSaoml9C6f95h0lbkdGsw4vFayK7xMvHUUFbzl3JtDf+mush4b4SzY40I017ZprX8cYovLuY2ZEui4+sOyGMPL1hPbIn1aP8UvZkEd45utSvs12zru8wVGfOf33dzQksqdR4le0pAvLGjvDUykAZOZxwtjLOo+G63VtUAT63IeCEuoUMKqI+pK6wuru4vy+fHqOCXK8hXsUI3S6Rjrnur3LQY3I9cvLaxFk4KGPyzAhCfCBxk8q684k4XKnBiBWhF4RCPLnZp5HghVCR9LtEEddMFv5QIvaMGzNdPHEVl/EKMwgUJkZTbUOVZeiePwp2rr7vOcAyo9eql9lMONJA1cmNuo0nT+7xRBcDvEMa1mc+uaBx7DU0qg/D6jFG/3OqXnEEvopzdBBYq1hmab3h5mAD434oCcor6sLXrhQYNMwYzKPGDnZ2fKD2fl+IBv3++YRVFBkN0PAzrJ4cWehNsAsdyy6qkDK2gVHf+yilwA/KSTgGdo9k7un0GbLXOtX7n5AWGReWx4MbQ0arIAk1jhgpF3AJbRE6uimfsyv5eSXLRpzSH2xkjmtBZ93zAQhdTGFAEE9yDdWnwSGWHujK883LpaG5Crh4FsFidm2JQhn6tsYz1ux2PiIEvQKarp+p9wZb6tV5dJVyrEipjli100awaX4NWufmEN77quBl9RmThQ5nr3wKO8Opz/Uirm9ZLw/4WCFQl5089V4swI+qUoYXWKXkrnys7PnOpMWSWOXue7UW/NNkog4maaBdLsJnmFBPgAbWU51jDw+UJxMmmjVUkR+JQNBU+4YiCN7hvguaQZeMEBKFb9Ll7kpsLayVVDCIs5X1sB5qe9LpG4ZT6yy+9IXMkVLUl3LhwmxihMP88Hy8Q+xwmCt8h8o8xBr+bmm6xSX1rGwnZIAdv18S8RbzoclVuMw1P7u2rNjqji4r7Jp0zn597/gkXuusrTeDXl1qlOwxgUTxmRNkBeRiXgBeOnedG0gCQVfhXh8Z9cZL2L0MHlC2xngBZROAGsI0vASVtQd+Ov/pw6/n/uomCalyODABVtHKEo++inZTihvwbTjAmdNeHc468KDbmRvUzD1fgXHmVonLWrdX4EKS289Is2vX4/zWOjBQVur+3trKTtfH6xGzQ7ZFWZ27rn1Dd2+MnQHs8vn0068/nytWU+OK2uKis8ejIqDyM5iiD7uUTXKGnYIsyzx9DPwTy97vOSvoIJp6Zl6IpcxXm21cXYGIxS1cm8KmqoZfhXtqhRpVGjd5AGFlLffiHg591TfiTPdK1WIJzuVKNl1fDr6bhL/jUMAWVc1dJFW6brivn5pqZNnlYM8h/9S4mh4sTm0mzjO0QDDc0/Z+34nV7JU6eDdoaFdCE+GrO/Ofe/umu+BIXkz9k6mCQq9zhe5zf/8MejeFXk2hN1PM5sZfixl9K+Yb8QshlwiwWpRUgOfd0BuvhAEVN7mPUk/HntNyw63A3YsSgKPykGG3UEdWfYvdwoQO6rQHUeS5mC7Yk+KV17Py3PwP+1FDd4+5zUs9DGE3siNngVJGZLlgb4wCt0DUtSTz23LE51CjNTleML150mJPhy+qdrh27nhT95PlHQG4n6zuCKf7yfoObQG7oHtnFAiayt2gw1cHaFTd2qWyP1I4tNjSMzeY5bldGOnf3Xt8SwWnVINyR3R5j9GyNfOB1udOJ3Z67B6pjTBnVxc7yfGMWZg0eMiwjmv+CzkU/RgGDZ2cTEFACjyLxmVlDFKlm1C7tzT7zQLBn1EreHaJy4YXVdfDZ9knZItNThW9wZWKroCwajki/vjY0QarnuAubP2kzFEhRmenTZpbbSxAmFrmlEDoUf6VvA2zeLtMYlHNe1h06fpIUWZhWfGeIg8Z3Q+emOlrh3KAidGkO5h1v2MtYHc+o3zR7xbublunuPbdQc/kwg7KYakwnOmEYA0m/Ct2vdOuqfhMoc8iIIdSY1BnC7bj7bJ4TLt0VsKF/Dq8u/8zgkmk7W3qK5BmvjjPeL1nDuDesZcqY9TxQ8O8MHxu+n6q8S3xxr9skgyG6/5biRU72erxfCEAaoP7idwj9AXvFsszrGYnU6vdhn8MQjdnDDpP+gnPViZpbE66ejXMXukRR7m0NQ5uyWCMtA0S0LyI6E186hiPyDRU2z5iyBIUZeDISz7CTTX8H1S5AB4+dg+I+ibFzi9leZS4WQva8Ae//POoRbj5OKpa7Fp2zjeVlJDgrzBLWLdZZ8lBVyX91tI+0+2bXy4S34evpt9CdJ8VN+QWUA5gF+KH0LiKYi3q9DMkAFuI/WAtLIHo7U1mr7qXt2pVZ9MPI+ZZ2IlMp4Vk0ffxnaVNq+kIGKPBjwCzZ1qzqYr2cgOZTETl7Oi6jrrjNrOFkykogjtA5WiwQxTH4S1meG9aJ6zajmI/dGjlYl3HVmcL9vWBWb/H1o22++hNImYo4MzN5UC0dnM1vzPFo6qJygE53q7Ej47ui/uOrmLqpX5pBiPgQLyxqvliJvDcoWYBB/k2pwPH0ym29dVt3QnhyXcBkFhQj+QE/8PfR0NfAGRvYvqRrmvQiJp/oiNwxvvS+tktM4qZhlsE2+p8Iz7s+XU8ZIKec8GbW/gO3h3/1TBN+7k4nn4bDFSV2iCPqO8RO54BuzpNpDA/wMYGPnBwAZuqzh/i1KAk/hCfSMTg4py5c4TXZ9oKaCfDPTt/IK5/TCYT/W9OF073+wK2W+x86HVouc/XZKDDa3ZZyg91PmxESDr/RTQ29SHYyl118RT97tPFPaB9d33xlFVGfad+TzKt/sz4BlaTp5OnPIWWfao3+tRg8pS96VMfRnow1LHYjC+TXhDnftJ6I1zkD3cnqx946w4d/gaP4hWWibER2ZuzJAjnX//6FwuS9ZpDp4kpvZUPwZWPAxWV+8J4dvrxw9l5sAUiBdwDS1Ue5/9yJ/g3iIlL6HrP8IYu/dKPL0YIRr97WYb6NxmDN9Vli5LxEb9VgAx3qGGjeBQlxSqKVHd7ECcJAqHRrlVPpfMCrLz6uotSv3O/O6ers37NLF1w3TdnD4KkuI4f65eWsRc7amCCM7oKV1weO9oueI5M2YOSrgxzioXWYywaXfjAthiUx6qljyKhyspEF/TmepoutIygzTXnnRnTUfiqSvxYmqgS8qNJaNWTv4KGuii+s4eT0VVMnXyXSqqQFIdlYOuIMW6Q23HKoEXd0YcCcKmEElLxGtxJg0V4TAb9S/ubfQahy5MxH1x/f4zOKOaOnO+n/IX3ph8p1L9/Pod9SfUrbEWd0u/5qLc39Q9s/QDJhPmlJYoh81xe8k//cOln0ONMmBCDabwbqyblJ1bdRLfbKCw9C48zmqlg1l3pS71yT+2H/ZcG4oMDWDOK0HxHEeQUUbTFxrbI6Q6kwh0+WOdWJgCie2zV7e77g/8Al34i0g=='}, 'src/reversibility/optimized.py': {'sha256': '4c0eeedf174cc1448b709e1b87d12ab7922e07fcccfe06867d5aee960a9485cb', 'compressed': 'eJylV91v2zYQf89fwWkvcuqoSQv0IZ2HAt2CdcO2Yuv6EgQCLZ4tIpKokJSbpOj/vjt+SJTtZgUmBI5N3vfd7+6UZdnP9z1o2UJnecOs5rKT3Zb13NbmNdto9Qgda5WA5rm7BM24BjZ0Vc27LYgiy7IT2fZKW1apzsK9beQ6nlilq3r2o+i6YoPcVqoOFXLDrk5QSxtu+WDVVnMx0rDArLoKSiE3G9BoquTrBjxfoWEH2kg8iLQbpT9xLcqeS71krRS9kp0t8biCk5OTquHGsL9gMCB+D5f5nvqroH5xecLweWMst7JqwdZKuBMBm6gnr+z9kvVLdrdk60ZVt2bJ6iU77bnmyIHGBSn0IG0Rieh7zVYT04xKwE5WgNd9+FrYhx5mJLzt8d6bLk1J1lfc2BI6io8zLPAu5qJJ0si5BTuxuqujjOitt5TJLpg8uUUPRQCFptHPHV0MThZTkaGvc4sM3wElqFzz6tYF1bOkQRwZNNhBd07oyVfS8+ZYvcS8jTpc4gTKEXdJioLmEn2JponSQmeUNiMRVYlEyQaprm/G40/S1kkpU0iTYC5ZyMwqZA81U7xXY1IW85DOYu4LPaTVJ2CPnJ7v2dtBk8+sZ+/+Zj1ySTUYdlewPztgDgRkfV8zA3oHBkU7ACHWeCfYx1/fFwdCE7e8ByX5nx9R7wIYdJZ3oXotr+p8gUi9G6QG45jL/IMekvLac9tV/gt2GkByugfjsbBGVcclIRV2mdV+d3HWO0FLlk8yMOtObjFVXb5YuPI4kD5yEQTv2Jk3enT2gN6XGdKKO6o59szbdn1+c0A61tb1+eX5DbJ4yovLQ9KAucmY0YA0Nk9YBQ0LcXBK9jHmrV6yP1QH8fN0tA+bKeGplkJAVxIIweRuWCyZFNjRNDXZ1RVvTKxsuaGbohOyZd9RhrHC6cDUvIfrixv2o582BQ6TjdwWLb8vDdYNEJ4b6La2nooOB5IB9pE3A/ystdJ59q7b8UYKrOl+sMwJfR7GUua975WR1NnNVBWaJlmeWoFOO8Cu6DBthPfI5e2z6hZ9hnYNQuDEJPYFJtVfRiXJ/ah3EeNQc8Ot1TFgWaU0ZAmkkMTFbw4yPPU6fLejIE59laLpb2OPK+mnI9KAgTBWD5XNDnF7GMorB0JnAYvIZYkQECMms3lhlUsXp/mELXjfNw85Nlz8CzaGued/GQt9aeQjVVh6PwfjqAia/chMuaFI5veedk53PKbf6PwRd2cNOrX60DR3Hq0K8PIcG4mrUNkp3dK1xxQOhzIUbh6HBwK6ktSocXVAOMPKddARV8mWwFaY8GoQPHM9fWR0hbDpX75IKiCYsje0PHecT5+R6eJVdhmoNo3i9uIV1ux6kxyvw/mX61HhTTC1HAcf/ZrFYNoai25omuh0DERVD90tDuBxaKf9peFrzC+htVMtRhHtWHqO0uETrwx2NdAr17qOjogpOisfmmNtC7dcmp5UWHHBeR4teo30DXCsFEDn2A4DuB4arh/CmJVtC0Jib2weCr+v/AbQI1TllhLPKq2MwQBZrfoHFjOAjaqR1UPBfgEu2CeQ29qiohZ3cMOsBPHaiarpdlpGuBCYDDa2nemqYD9JQznwLE4FJYOIrHKy+E5h6yTn6ZDj9g8QKL0n3JnqA2yo19ga2FqJh3GlKmK0YlGmyWA/rNg5sSXpcmdPdfS3JIBRX3ClTCjBDv/I3ctBKqgd0M41hAa/g4BSt7qkaJpaeoKopO36aYaA/Y+x5sVj4jcjbbrquFGSn10s452fLWcXN08sQpZr3MZpNPnKTuQEdVQqVtlkqXkETEpOA/9gZkX4Jqh9+cILosaFnuHbEm2WbgCeI56gy4MNizmQkgB9W0Rjj0pRFJ9GbaXz8qpo8JWS65zCeO3sufRWPZtpv4lDomlLKt/C42GxJ9V4mTNA5V7ZMsb2aS1yiwUGpewE3K/OLs7PKdlicO+Cq8wM7d6oi+WIOFl5A56nRTmjzRNiRIZvTEwat1a5ORXOCvcvIV8sxnme7HBJLTzzyqf6c7lOaGnPi4EgymVi+f+ByWQXZdDt1bFmS25yT7SY9fvJ7Njh+6E0He9NrWy+I+hPTfcdJqIH/MD3mbfv/2HhNewS8YELAVO9lS36QLUMPXYm3nXKYisgzGIHi2KLpCOhI7jBcNwnvbJlwNEHJ3pxMBgd0RRZq3A29gNOiQqrazVh96hoISt7KPLz7eWe1wsHSHyh2REevUppocWV58vXpTfSHJF+fUz0JPfmiUAMfQOHAt1x/rTUWYrdycm/2y/bzw=='}}
for relative, entry in EMBEDDED.items():
    payload = zlib.decompress(base64.b64decode(entry['compressed']))
    if hashlib.sha256(payload).hexdigest() != entry['sha256']:
        raise RuntimeError('Embedded checksum mismatch: ' + relative)
    destination = REPO / relative
    if destination.exists():
        raise RuntimeError('Refusing to replace an existing source: ' + relative)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(payload)
BENCHMARK_SCRIPT = REPO / 'scripts/benchmark_optimized.py'
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
RECEIPTS = OUTPUT.parent / (RUN_ID + '-receipts.json')
RECEIPTS.write_text(json.dumps({'source_commit': PINNED_COMMIT,
    'embedded_sha256': {p: v['sha256'] for p, v in EMBEDDED.items()},
    'runtime': actual, 'declared_environment_differences': mismatches,
    'data': receipt, 'probe_only': True, 'fixed_reserved_headroom': .10}, indent=2) + '\n')
print('Tested optimization harness staged:', BENCHMARK_SCRIPT)


## Run the gated experiments

Every benchmark runs in a fresh process with 5 warmup and 20 timed complete updates. The runner continues after OOM or a failed candidate and preserves logs. Numerical failure blocks the optimized sweep. Fused AdamW is considered only if its independent update gate passes.

Experiments: six baseline-29 reference probes; shared chunk sizes 1024/2048/4096 at batches 29/40/50; midpoint chunk-only ablation; fused optimizer alternatives at batch 50; three repeats of each selected batch-50 finalist. The final repeat writes a CPU checkpoint and separate `checkpoint-overhead.json` outside training timing. No allocator setting or headroom relaxation is used.


In [ ]:
WARMUP_UPDATES = 5
TIMED_UPDATES = 20
command = [sys.executable, "-u", str(BENCHMARK_SCRIPT), "--suite", "--config", str(reference_config),
           "--data-dir", str(LOCAL_DATA), "--output", str(OUTPUT),
           "--warmup", str(WARMUP_UPDATES), "--updates", str(TIMED_UPDATES)]
with (OUTPUT.parent / (RUN_ID + "-orchestration.log")).open("w") as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    returncode = process.wait()
print("Exit code:", returncode)
print("Evidence:", OUTPUT)
if returncode:
    print("One or more passes failed. Continue to the next cell to inspect preserved logs and results.")


In [ ]:
from IPython.display import display, Markdown
report = OUTPUT / 'REPORT.md'
if report.exists():
    display(Markdown(report.read_text()))
suite = json.loads((OUTPUT / 'suite.json').read_text())
print(json.dumps(suite.get('decisions', {}), indent=2))
for row in suite['records']:
    if row['result']['status'] != 'complete':
        print('FAILED:', row['name'], row['result'].get('error'))
        print((OUTPUT / (row['name'] + '.log')).read_text()[-4000:])
print('CPU checkpoints stay in Drive under finalist folders; do not resume a full run from probe snapshots.')


## Export evidence

The compact ZIP includes results, the experiment report, receipts, all logs and separate checkpoint-overhead records. CPU checkpoint binaries remain in Drive under the finalist folders; their paths, sizes and SHA-256 hashes are in the records. Export preserves failures too.

A batch-50 failure or speed regression is an experimental result. Review this ZIP before committing any full-training budget or calculating monetary savings. Custom fused vocabulary kernels and CPU activation offload remain later experiments if measured results justify them.


In [ ]:
import zipfile
BUNDLE = OUTPUT.parent / (RUN_ID + '-optimization.zip')
if BUNDLE.exists():
    raise RuntimeError('Export already exists; preserved existing evidence')
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and path.suffix != '.pt':
            archive.write(path, arcname=str(path.relative_to(OUTPUT.parent)))
    archive.write(RECEIPTS, arcname=RECEIPTS.name)
    archive.write(OUTPUT.parent / (RUN_ID + '-orchestration.log'), arcname=RUN_ID + '-orchestration.log')
print('Evidence:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
